# 02 — Leakage-Free Feature Engineering & Preprocessing Pipeline

Demonstrates the reusable `TelcoFeatureEngineer` + `ColumnTransformer` pipeline from `src/feature_engineering.py`, verifying strict train/test separation, zero NaNs after transformation, and deterministic feature ordering.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.config import PROCESSED_DATA_PATH, RAW_FEATURE_COLUMNS, TARGET_COL, RANDOM_STATE, TEST_SIZE
from src.feature_engineering import TelcoFeatureEngineer, build_preprocessor, get_transformed_feature_names

df = pd.read_csv(PROCESSED_DATA_PATH)
X = df[RAW_FEATURE_COLUMNS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape} (churn={y_train.mean():.3%}) | Test: {X_test.shape} (churn={y_test.mean():.3%})')

Train: (5634, 19) (churn=26.535%) | Test: (1409, 19) (churn=26.544%)


In [2]:
# Inspect engineered features before encoding
fe = TelcoFeatureEngineer()
eng_sample = fe.fit_transform(X_train.head(5))
eng_sample[['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'charge_ratio', 'service_count', 'has_support_bundle', 'is_auto_pay', 'tenure_group']]

,tenure,MonthlyCharges,TotalCharges,avg_monthly_spend,charge_ratio,service_count,has_support_bundle,is_auto_pay,tenure_group
3738,35,49.20,1701.65,48.618571,1.011959,4.0,0,0,25-48m
3151,15,75.10,1151.55,76.770000,0.978247,3.0,1,0,13-24m
4860,13,40.55,590.35,45.411538,0.892945,4.0,1,0,13-24m
3867,26,73.50,1905.70,73.296154,1.002781,6.0,0,1,25-48m
3810,1,44.55,44.55,44.550000,1.000000,2.0,0,0,0-12m


In [3]:
# Fit preprocessor strictly on X_train, then transform X_test
preprocessor = build_preprocessor(scale_numeric=True)
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
feature_names = get_transformed_feature_names(preprocessor)
print('Transformed Train Shape:', X_train_t.shape)
print('Transformed Test Shape:', X_test_t.shape)
print('Any NaN/Inf in Test:', not np.isfinite(X_test_t).all())
print('First 15 transformed features:', feature_names[:15])

Transformed Train Shape: (5634, 54)
Transformed Test Shape: (1409, 54)
Any NaN/Inf in Test: False
First 15 transformed features: ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'charge_ratio', 'service_count', 'SeniorCitizen', 'has_support_bundle', 'is_auto_pay', 'gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes']
